# Cast / Sight — Complete ML Pipeline
Real casting data, executed baseline, evaluation, explainability, and business interpretation.

## Scope
This notebook consolidates the saved Phase 1–7 outputs. It reloads measured artifacts; it does not invent results or retrain the model.

In [1]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display
root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd(); print('Project:',root)

Project: C:\Users\AbdElhalk\OneDrive\Desktop\computer vision\Industrial Quality Inspection


## Dataset audit
The manifest is the leakage-safe source for all downstream work.

In [2]:
manifest=pd.read_csv(root/'data/processed/phase2_clean_splits.csv')
print('Rows:',len(manifest),'Unique paths:',manifest.path.nunique())
display(manifest.groupby(['split','label']).size().unstack(fill_value=0))
print('Duplicate paths:',manifest.path.duplicated().sum())

Rows: 7284 Unique paths: 7284


label,Defective,OK
split,,
test,632,461
train,2947,2151
validation,632,461


Duplicate paths: 0


## Preparation
Duplicates were removed before splitting; validation and test transforms stayed deterministic.

In [3]:
prep=json.load(open(root/'data/processed/phase2_preprocessing_config.json'))
print('Seed:',prep.get('seed',42),'Input:',prep.get('image_size',[224,224]))
print('Train augmentation:',prep.get('train_augmentation','training-only'))
print('Cross-split duplicate hashes:',prep.get('cross_split_duplicate_hashes',0))

Seed: 42 Input: [224, 224]
Train augmentation: training-only
Cross-split duplicate hashes: 0


## Baseline training
The saved MobileNetV2 head baseline used transfer learning on the prepared train/validation sets.

In [4]:
history=pd.read_csv(root/'results/phase3_training_history.csv')
display(history)
print('Training curves are preserved in the original phase notebook; visual artifacts are excluded from this clean export.')
print('Checkpoint: phase3_mobilenetv2_head_baseline.pt')

,epoch,train_loss,train_accuracy,train_precision,train_recall,train_f1,val_loss,val_accuracy,val_precision,val_recall,val_f1
0,1,0.484600,0.795214,0.786338,0.886664,0.833493,0.377998,0.864593,1.000000,0.765823,0.867384
1,2,0.306240,0.903295,0.913131,0.920258,0.916681,0.289176,0.903934,1.000000,0.833861,0.909405
2,3,0.250439,0.917615,0.924136,0.934170,0.929126,0.221585,0.949680,0.989813,0.922468,0.954955


Training curves are preserved in the original phase notebook; visual artifacts are excluded from this clean export.
Checkpoint: phase3_mobilenetv2_head_baseline.pt


## Test evaluation
All values below come from the untouched test prediction file and saved evaluation JSON.

In [5]:
metrics=json.load(open(root/'results/phase4_complete_metrics.json'))
summary=pd.DataFrame([{k:metrics[k] for k in ['accuracy','precision','recall','f1','roc_auc']}])
display(summary.rename(columns={'accuracy':'Accuracy','precision':'Precision','recall':'Recall','f1':'F1','roc_auc':'ROC-AUC'}))
display(pd.DataFrame(metrics['classification_report']).T)

,Accuracy,Precision,Recall,F1,ROC-AUC
0,0.940531,0.993043,0.903481,0.946147,0.993503


,precision,recall,f1-score,support
OK,0.882239,0.991323,0.933606,461.000000
Defective,0.993043,0.903481,0.946147,632.000000
accuracy,0.940531,0.940531,0.940531,0.940531
macro avg,0.937641,0.947402,0.939877,1093.000000
weighted avg,0.946309,0.940531,0.940858,1093.000000


## Confusion and predictions
The failure cases remain available for inspection rather than being hidden by aggregate scores.

In [6]:
pred=pd.read_csv(root/'results/phase4_test_predictions.csv')
print(pred.case.value_counts())
print('Confusion/ROC visual artifacts are excluded from this clean export; numeric results remain in results/.')
display(pred[pred.case!='Correct'].head(10))

Correct           1028
False negative      61
False positive       4
Name: case, dtype: int64
Confusion/ROC visual artifacts are excluded from this clean export; numeric results remain in results/.


,path,actual,predicted,defective_score,case
54,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.381947,False negative
65,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.484779,False negative
67,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.368446,False negative
70,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.452545,False negative
94,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.303770,False negative
95,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.489236,False negative
121,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.461636,False negative
122,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.333822,False negative
139,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.463561,False negative
152,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,1,0,0.327389,False negative


## Controlled experiments
Phase 5 compared two one-pass improvements against the same unseen test set.

In [7]:
comparison=pd.read_csv(root/'results/phase5_model_comparison.csv')
display(comparison.style.format({c:'{:.4f}' for c in comparison.columns[1:]}))
selected=json.load(open(root/'results/phase6_selected_model.json'))
print('Selected checkpoint:',selected['source_checkpoint'])

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Original MobileNetV2 head,0.9405,0.9930,0.9035,0.9461,0.9935
1,112px unweighted head,0.6551,0.9381,0.4320,0.5915,0.9175
2,112px class-weighted head,0.4959,1.0000,0.1282,0.2272,0.8924


Selected checkpoint: phase3_mobilenetv2_head_baseline.pt


## Explainability
Grad-CAM highlights spatial evidence for correct and incorrect examples; it is decision support, not proof.

In [8]:
cam_cases=pd.read_csv(root/'results/phase6_gradcam_cases.csv')
display(cam_cases[['path','case','label','predicted','defective_score']])
print('Grad-CAM visual artifacts are excluded from this clean export; cases remain in results/.')
print('Explanation method: MobileNetV2 final convolutional feature block')

,path,case,label,predicted,defective_score
0,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,Correct,OK,0,0.064501
1,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,Correct,Defective,1,0.625515
2,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,False positive,OK,1,0.549593
3,C:\Users\AbdElhalk\OneDrive\Desktop\computer v...,False negative,Defective,0,0.381947


Grad-CAM visual artifacts are excluded from this clean export; cases remain in results/.
Explanation method: MobileNetV2 final convolutional feature block


## Business insights
High precision reduces unnecessary defect escalations; lower defective recall means missed-defect review remains important.
Inspectors should use Grad-CAM and the original image together before release, and monitor false negatives in production.
The measured comparison selected the original 224px baseline; tested 112px variants were not promoted.

In [9]:
insights={'test_images':len(pred),'false_positives':int((pred.case=='False positive').sum()),'false_negatives':int((pred.case=='False negative').sum()),'selected_model':selected['source_checkpoint']}
print(insights)
print('Pipeline artifact review complete; inference is served by fastapi_backend/main.py.')

{'test_images': 1093, 'false_positives': 4, 'false_negatives': 61, 'selected_model': 'phase3_mobilenetv2_head_baseline.pt'}
Pipeline artifact review complete; inference is served by fastapi_backend/main.py.
